# Generate a 2D spectral target profile

This notebook evaluates the reference coupling-dressed Green kernel 
`F_{l,jk}=d_j G_{jk}(omega_l)d_k` and stores the sampled spectral target profile in CSV format.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

resol = 160
plt.rcParams.update({
    "font.family": "serif",
    "axes.linewidth": 0.7,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
})

cwd = Path().resolve()
project_root = next(path for path in (cwd, *cwd.parents) if (path / "src").exists())
sys.path.append(str(project_root))

from src.fiber_geometry import load_geometry, plot_geometry
from src.target_profile import (
    generate_target_profile,
    make_frequency_grid,
    save_target_profile_csv,
)


Load the geometry. The CSV only provides atom positions and box metadata; `Omega_j` and `d_j` are specified below as experiment parameters.


In [ ]:
kind = "slab"  # "slab" or "disk"
lengths = [50.0, 1.0]
matter_width = 1.0
matter_height = 1.0
N_A = 100
N_layer = 1  # ignored for disk geometries

geometry = load_geometry(kind, lengths, matter_width, matter_height, N_A, N_layer)
atom_positions = geometry["atom_positions"]
L1, L2 = geometry["lengths"]

print(f"Loaded geometry from: {geometry['path']}")
print(f"N_A={geometry['N_A']}, N_layer={geometry['N_layer']}, lengths={geometry['lengths']}, kind={geometry['kind']}")


In [ ]:
fig, ax = plt.subplots(figsize=(8.0, 2.2), dpi=resol)
plot_geometry(kind, lengths, matter_width, matter_height, N_A, N_layer, ax=ax)
ax.set_title(f"Loaded geometry: {geometry['label']}")
plt.show()


Choose the reference window, the uniform reference atom parameters, and the finite frequency calibration set.

The CSV stores one row for each `(omega_l, j, k)` component of the spectral kernel.


In [ ]:
Omega = 10 * np.pi
d = 1.0
Omega_j = Omega * np.ones(geometry["N_A"])
d_j = d * np.ones(geometry["N_A"], dtype=float)

reference_cutoffs = {
    "omega_min": 0 * np.pi,
    "omega_max": 20 * np.pi,
}

nb_omega = 50
omega_values = np.linspace(Omega - 2 * abs(d) ** 2, Omega + 2 * abs(d) ** 2, nb_omega)

frequency_set = make_frequency_grid(omega_values)
radius_cache_decimals = 4 if geometry["kind"] == "slab" else 12

print(f"N_omega = {len(frequency_set)}")
print(f"N_spectral_entries = {len(frequency_set) * geometry['N_A'] ** 2}")
print(f"W_omega = {frequency_set['W_omega'].iloc[0]:.6g}")
print(f"W_ljk = {frequency_set['W_omega'].iloc[0] / geometry['N_A'] ** 2:.6g}")
print(f"radius_cache_decimals = {radius_cache_decimals}")
frequency_set.head()


In [20]:
target_profile = generate_target_profile(
    frequency_set,
    atom_positions=atom_positions,
    Omega_j=Omega_j,
    d_j=d_j,
    reference_cutoffs=reference_cutoffs,
    epsabs=1e-7,
    epsrel=1e-7,
    cache_decimals=radius_cache_decimals,
    progress=True,
)

target_profile.head()


target spectral profile:   0%|          | 0/50 [00:00<?, ?omega/s]

,omega,j,k,W,F_real,F_imag,reference_Omega,reference_d
0,29.415927,0,0,0.000002,-0.141117,-0.058411,31.415927,1.0
1,29.415927,0,1,0.000002,0.210746,-0.019347,31.415927,1.0
2,29.415927,0,2,0.000002,0.071217,0.011300,31.415927,1.0
3,29.415927,0,3,0.000002,-0.031258,0.026464,31.415927,1.0
4,29.415927,0,4,0.000002,-0.081652,0.025164,31.415927,1.0


In [ ]:
window_label = (
    f"ref_{reference_cutoffs['omega_min'] / np.pi:g}pi_"
    f"{reference_cutoffs['omega_max'] / np.pi:g}pi"
).replace(".", "p")
output_dir = project_root / "2d_setup" / "generate_target_profile"
output_path = output_dir / f"target_profile_{geometry['label']}_{window_label}.csv"

save_target_profile_csv(target_profile, output_path)
print(f"Saved target profile to: {output_path}")
